# Fair Lending Screener — Demonstration Notebook

**Package:** `fair-lending-screener` v0.2.0  
**Methodology:** FFIEC Interagency Fair Lending Examination Procedures (2009)

---

`fair-lending-screener` applies the same adjusted denial disparity methodology federal examiners use — binary logistic regression with FFIEC-standard controls — to publicly available HMDA mortgage data. It produces an adjusted odds ratio, 95% confidence interval, p-value, and a Markdown report written for non-statistician audiences.

This tool is designed for **community advocates, investigative journalists, CDFI practitioners, and fair housing researchers** who need to evaluate denial rate disparities without access to $50K+ commercial compliance software or a Stata license.

Every result is a **screening signal, not a finding of discrimination.** Public HMDA data does not include credit score, AUS recommendations, or asset data — key underwriting factors whose absence means this analysis is an upper-bound estimate of the unexplained disparity. Federal examiners with access to full loan files may reach a different conclusion.

> **Alpha release (v0.2.0).** Methodology peer review by an external fair lending expert is planned before v1.0.0. Use as a screening tool to identify cases warranting further analysis, not as a basis for enforcement or accusation.

## 1. Setup

Install `fair-lending-screener` if it isn't already. Both import aliases resolve to the same package.

In [1]:
%pip install fair-lending-screener

You should consider upgrading via the '/private/tmp/notebook_test/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
import fair_lending_screener as fls
import fairlendingscreener          # alias — identical to fair_lending_screener

print(f"fair-lending-screener version: {fls.__version__}")
print(f"Methodology doc path:          {fls.get_methodology_path()}")
print(f"Limitations doc path:          {fls.get_limitations_path()}")

fair-lending-screener version: 0.2.0
Methodology doc path:          /private/tmp/notebook_test/lib/python3.9/site-packages/fair_lending_screener/_methodology_doc.md
Limitations doc path:          /private/tmp/notebook_test/lib/python3.9/site-packages/fair_lending_screener/_limitations_doc.md


## 2. Fetch HMDA Data

The Home Mortgage Disclosure Act (HMDA, 12 U.S.C. § 2801) requires covered lenders to report every mortgage application they receive, including the applicant's race and ethnicity, income, loan amount, debt-to-income ratio, loan-to-value ratio, geography (MSA/census tract), and the lender's action (originated, denied, withdrawn, etc.). The CFPB publishes the full Loan Application Register (LAR) annually — roughly 7–8 million records per year — making it the primary public data source for fair lending analysis.

We fetch 2023 data for Illinois, which covers the Chicago-Naperville-Elgin MSA along with smaller Illinois markets. MSA fixed effects in the regression model handle geographic variation within the state.

**Data fetch is backed by `hmda-analyzer`** (a dependency installed with `fair-lending-screener`). The `fls.load_from_api()` wrapper adds an endpoint health check, column normalization, and typed error handling on top of the raw `hmdaanalyzer` call.

> **Note:** This cell fetches data from the CFPB HMDA API. It requires network access and may take 1–2 minutes.

In [4]:
# Requires network access — tested by maintainer locally

# Verify the CFPB API is reachable before starting a long fetch
health = fls.check_data_source()
print(f"CFPB HMDA API: reachable (HTTP {health['status_code']})")

# Load 2023 Illinois HMDA data
# load_from_api() wraps hmdaanalyzer.load_from_api() with a health check and column normalization
print("Fetching 2023 Illinois HMDA data (this may take 1–2 minutes)...")
df_raw = fls.load_from_api(
    year=2023,
    state="IL",
    limit=50_000,
)
print(f"Raw records loaded: {len(df_raw):,}")

# Apply FFIEC-standard dataset filters:
#   - Conventional first-lien home purchase loans only (loan_type=1, lien_status=1, loan_purpose=1)
#   - Site-built one-to-four unit principal residence
#   - LTV ≤ 100%; action_taken in {1 = originated, 3 = denied}
#   - Log-transforms income, loan_amount, property_value; bins DTI
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df = fls.prepare_for_analysis(df_raw)

print(f"Records after FFIEC filters: {len(df):,}")

# Raw (unadjusted) denial rates by race — before any controls are applied
raw_denial = (
    df.groupby("derived_race")["is_denied"]
    .agg(denial_rate="mean", n="count")
    .assign(denial_rate=lambda x: (x["denial_rate"] * 100).round(1))
    .sort_values("denial_rate", ascending=False)
)
print("\nRaw denial rates by race (uncontrolled — no income, LTV, or MSA adjustment):")
raw_denial

CFPB HMDA API: reachable (HTTP 405)
Fetching 2023 Illinois HMDA data (this may take 1–2 minutes)...
Fetching HMDA data from CFPB API (year=2023, limit=50,000)...
Loaded 50,000 LAR records
Raw records loaded: 50,000
Records after FFIEC filters: 13,808

Raw denial rates by race (uncontrolled — no income, LTV, or MSA adjustment):


,denial_rate,n
derived_race,,
Free Form Text Only,100.0,1
2 or more minority races,33.3,12
Black or African American,22.9,751
Native Hawaiian or Other Pacific Islander,20.0,15
American Indian or Alaska Native,11.9,42
Race Not Available,8.1,1503
Asian,7.9,1524
White,5.3,9616
Joint,3.5,344


## 3. Run the Analysis

`adjusted_denial_disparity()` runs a binary logistic regression — specifically `statsmodels.api.Logit`, not scikit-learn or any black-box ML model. The outcome is denial (1) vs. origination (0). The coefficient on the protected-class indicator is exponentiated to produce the adjusted odds ratio.

**What "adjusted" means:** The regression controls for legitimate underwriting factors that are available in public HMDA data, asking the question: *among applicants who look similar on paper — same income bracket, similar loan-to-value ratio, same metropolitan area — do denial rates still differ by race?* The factors controlled for are:

| Control | Notes |
|---|---|
| `log(applicant_income)` | Ability-to-repay; log-transformed for right skew |
| `log(loan_amount)` | Loan size |
| `loan_to_value_ratio` | Collateral coverage |
| DTI bins (≤35%, 36–42%, 43–49%, ≥50%, missing) | Per The Markup (2021) / FFIEC binning |
| `log(property_value)` | Collateral value |
| MSA fixed effects (~300+ dummy variables) | Local market conditions, lender mix, home prices |

**Why `data_year` is required:** Years before 2018 lack `loan_to_value_ratio`, `debt_to_income_ratio`, and `property_value` — the controls that make the FFIEC model meaningful. Omitting them would produce an analysis that looks adjusted but isn't.

**Regulatory basis:** FFIEC Interagency Fair Lending Examination Procedures (2009) — the methodology OCC, Federal Reserve, FDIC, NCUA, and CFPB examiners use.

In [5]:
# Requires network access — tested by maintainer locally

result = fls.adjusted_denial_disparity(
    df,
    protected_class="Black or African American",   # derived_race value per Regulation C
    comparison_class="White",                       # reference group for the odds ratio
    data_year=2023,                                 # required; controls LTV/DTI/property_value availability
)

print("Analysis complete.")
print(f"Sample: {result.sample_size:,} applications ({result.sample_size_protected:,} {result.protected_class}, "
      f"{result.sample_size_comparison:,} {result.comparison_class})")

Analysis complete.
Sample: 10,367 applications (751 Black or African American, 9,616 White)


/var/folders/5s/8wtcjv3d7rq8w54bqxtv77pm0000gn/T/ipykernel_56442/3902041019.py:3: UserWarning: Dropping 1 zero-variance control(s) before fitting — they are constant in this sample and would cause a singular Hessian: ['dti_ge50']. Common cause: 'dti_missing' when no DTI values are missing, or an MSA dummy covering only one outcome category. Results use the remaining controls.
  result = fls.adjusted_denial_disparity(


## 4. Interpret the Result

The `DisparityResult` dataclass carries every number needed to evaluate, cite, and reproduce the analysis. Here's what each field means in plain English:

| Field | What it means |
|---|---|
| `adjusted_odds_ratio` | The core finding. 1.8× means Black applicants faced 80% higher odds of denial than White applicants *after controlling for income, LTV, DTI, property value, and MSA.* |
| `unadjusted_odds_ratio` | Raw denial rate disparity with no controls. The gap between unadjusted and adjusted shows how much income, LTV, and geography explain. |
| `p_value` | Probability this disparity is due to chance. Both conditions must hold for significance: p < 0.05 AND the confidence interval must exclude 1.0. |
| `confidence_interval_95` | The range of adjusted odds ratios consistent with the data at 95% confidence. If this range excludes 1.0 and p < 0.05, the disparity is statistically significant. |
| `is_statistically_significant` | Summary flag — `True` only when both conditions hold simultaneously. A large odds ratio with p = 0.08 is not flagged as significant. |
| `controls_used` | Columns that entered the regression after the zero-variance check. |
| `dropped_controls` | Columns excluded before fitting because they were constant in this sample (e.g., `dti_missing` when no applicant had missing DTI). A constant column would cause a singular Hessian. |
| `interpretation` | Auto-generated journalist-legible summary. This is the field to quote. It includes the appropriate disclaimer (significant vs. non-significant) and never uses the word "discrimination" as a finding. |
| `limitations` | The complete FFIEC-derived list of what public HMDA cannot tell you — bundled with every result so it can't be separated from the numbers. |
| `provenance` | Package version, dependency versions, timestamp, and input parameters. Copy this block verbatim when citing this result for reproducibility. |

In [6]:
# Requires network access — tested by maintainer locally

print("=" * 60)
print("CORE STATISTICAL OUTPUTS")
print("=" * 60)
print(f"Unadjusted odds ratio:      {result.unadjusted_odds_ratio:.4f}×")
print(f"Adjusted odds ratio:        {result.adjusted_odds_ratio:.4f}×")
print(f"95% confidence interval:    {result.confidence_interval_95[0]:.4f}× – {result.confidence_interval_95[1]:.4f}×")
print(f"p-value:                    {result.p_value:.6f}")
print(f"Statistically significant:  {result.is_statistically_significant}")

print()
print("=" * 60)
print("MODEL METADATA")
print("=" * 60)
print(f"Controls used ({len(result.controls_used)}):")
for ctrl in result.controls_used:
    print(f"  - {ctrl}")
if result.dropped_controls:
    print(f"Dropped (zero variance):  {result.dropped_controls}")
print(f"MSA fixed effects:          {result.model_diagnostics['n_msa_dummies']} dummy variables")
print(f"McFadden pseudo-R²:         {result.model_diagnostics['pseudo_r2_mcfadden']:.4f}")
print(f"  (Real HMDA: expect 0.15–0.25 per The Markup 2021 calibration)")
print(f"Model converged:            {result.model_diagnostics['converged']}")

print()
print("=" * 60)
print("INTERPRETATION (auto-generated — quote this field)")
print("=" * 60)
print(result.interpretation)

print()
print("=" * 60)
print("LIMITATIONS (first 3 of full list)")
print("=" * 60)
for i, lim in enumerate(result.limitations[:3], 1):
    print(f"{i}. {lim}")
print(f"  ... ({len(result.limitations)} total — all included in the full report)")

CORE STATISTICAL OUTPUTS
Unadjusted odds ratio:      5.2605×
Adjusted odds ratio:        4.8057×
95% confidence interval:    3.8804× – 5.9517×
p-value:                    0.000000
Statistically significant:  True

MODEL METADATA
Controls used (7):
  - log_income
  - log_loan_amount
  - loan_to_value_ratio
  - log_property_value
  - dti_36_42
  - dti_43_49
  - dti_missing
Dropped (zero variance):  ['dti_ge50']
MSA fixed effects:          0 dummy variables
McFadden pseudo-R²:         0.1367
  (Real HMDA: expect 0.15–0.25 per The Markup 2021 calibration)
Model converged:            True

INTERPRETATION (auto-generated — quote this field)
Analysis of 10,367 applications (Black or African American vs. White) found a statistically significant adjusted disparity (adjusted odds ratio: 4.81×, 95% CI: 3.88–5.95, p=0.0000). Unadjusted denial rate disparity: 5.26×. Black or African American applicants faced 4.81× higher odds of denial than White applicants after controlling for 7 legitimate underw

## 5. Generate the Full Report

`generate_disparity_report()` produces a Markdown document designed for inclusion in internal memos, board presentations, or regulatory submissions. It includes:

- Headline finding with the adjusted odds ratio
- Key numbers table (unadjusted vs. adjusted, CI, p-value, sample sizes, pseudo-R²)
- Explanation of what "adjusted" means
- The complete list of controls used and any dropped controls
- Full limitations section
- "What This Does NOT Prove" section
- Methodology citations (FFIEC, The Markup, Wooldridge OVB)
- Provenance block for reproducibility

**Lender-name safety guardrail:** If a `lender_name` is passed, it is suppressed from the headline when the analysis doesn't meet quality standards — specifically when p > 0.05, pseudo-R² < 0.05, or the model didn't converge. The numbers are still reported, but the report flags the quality issue explicitly. This prevents the tool from producing lender-attributed headlines on low-confidence analyses.

In [7]:
# Requires network access — tested by maintainer locally

# lender_name is omitted here — the demo notebook does not name specific lenders.
# When used with real lender data (filtered by LEI), pass lender_name only if the
# result is statistically significant — the guardrail will suppress it otherwise.
report = fls.generate_disparity_report(
    result,
    geography="Illinois",
    year=2023,
    include_methodology=True,
)

try:
    from IPython.display import Markdown, display
    display(Markdown(report))
except ImportError:
    print(report)

# Fair Lending Disparity Analysis — Illinois — 2023

*Generated: 2026-05-24 02:32 UTC*

> **Alpha release notice:** Alpha release (v0.2.0). Methodology peer review by external fair lending expert planned before v1.0.0. Use as a screening tool to identify cases warranting further analysis, not as a basis for enforcement or accusation.

## Headline Finding

After controlling for 7 legitimate underwriting factors, **Black or African American applicants faced 4.81× higher odds of denial** than White applicants. This disparity is statistically significant (p=0.0000).

> This analysis identifies a statistically significant adjusted disparity in denial rates. It does not constitute a finding of discrimination under ECOA or the Fair Housing Act, and it does not establish that any protected class characteristic caused any lending outcome. Further review of application files, underwriting guidelines, and internal lender data would be required to assess whether discrimination occurred.

## Key Numbers

| Metric | Value |
|---|---|
| Unadjusted odds ratio | 5.2605× |
| **Adjusted odds ratio** | **4.8057×** |
| 95% Confidence interval | 3.8804× – 5.9517× |
| P-value | 0.000000 |
| Statistically significant? | Yes (p < 0.05, CI excludes 1.0) |
| Total applications analyzed | 10,367 |
| Black or African American applicants | 751 |
| White applicants | 9,616 |
| McFadden pseudo-R² | 0.1367 |
| Model converged | Yes |

## What 'Adjusted' Means

The **unadjusted** odds ratio compares raw denial rates between groups, without accounting for any differences in income, loan size, or other factors. The **adjusted** odds ratio accounts for the legitimate underwriting factors listed below — it asks: among applicants who look similar on paper (same income bracket, similar loan-to-value ratio, same MSA), do denial rates still differ by race?

The difference between the unadjusted and adjusted figures shows how much of the raw disparity (5.26×) is explained by the available controls. The remaining adjusted disparity (4.81×) persists after accounting for those factors.

## Controls Used in the Adjusted Model

The following legitimate underwriting factors were included as controls in the logistic regression model (FFIEC Interagency Fair Lending Examination Procedures, 2009):

- `log_income`
- `log_loan_amount`
- `loan_to_value_ratio`
- `log_property_value`
- `dti_36_42`
- `dti_43_49`
- `dti_missing`

**Controls dropped (zero variance in this sample):** `dti_ge50`. These columns were constant in the analysis sample — they contribute nothing to model fit and would cause a singular Hessian. Common cause: `dti_missing` when all applicants reported DTI, or an MSA dummy covering a single outcome category.

## Limitations of This Analysis

**Public HMDA data does not include several factors that affect lending decisions.** This analysis controls for what is available, but cannot control for what is absent. Each omission is a source of omitted-variable bias.

- No credit score in public HMDA data — the single most predictive underwriting variable (excluded by industry during Dodd-Frank rulemaking; 12 C.F.R. § 1003.4(a)(15) note).
- No AUS recommendation in public HMDA data — Fannie/Freddie DU/LP decisions, the primary conventional loan underwriting tool, are not in the public file.
- No asset or reserve data — liquid assets, retirement accounts, and reserves are not reported and are a legitimate underwriting factor.
- No co-applicant credit profile — co-applicant income is reported but co-applicant credit score, employment, and liabilities are not.
- No credit history or tradeline data — derogatory marks, delinquency history, and charge-offs are significant underwriting factors absent from HMDA.
- No underwriter override data — AUS approvals overridden to denial (or vice versa) are invisible in HMDA; these discretionary decisions are a known vector for bias.
- No appraisal method flag — property value is lender-reported; appraisal gaps in majority-minority neighborhoods are a documented issue not controllable here.
- No employment history — employment type (W-2 vs. self-employed), tenure, and industry are underwriting factors not reported in HMDA.
- Omitted-variable bias direction: omitting AUS and credit score, both of which correlate with race and denial, produces a known upward bias in the racial disparity coefficient (Wooldridge 2019, § 3.3). The adjusted odds ratio from this tool is an upper bound estimate relative to a fully specified model.

## What This Analysis Does NOT Prove

This analysis is a **statistical screening tool**, not a legal finding. Specifically:

- It does **not** prove that the lender intentionally discriminated.
- It does **not** establish a causal link between race and denial.
- It does **not** account for credit score, AUS recommendations, or asset data.
- It does **not** constitute a finding of a violation of ECOA or the Fair Housing Act.
- It does **not** mean any specific applicant was treated unlawfully.
- It **does** identify a statistical pattern that warrants further review with full loan-file data, internal underwriting guidelines, and examiner access.

> This analysis identifies a statistically significant adjusted disparity in denial rates. It does not constitute a finding of discrimination under ECOA or the Fair Housing Act, and it does not establish that any protected class characteristic caused any lending outcome. Further review of application files, underwriting guidelines, and internal lender data would be required to assess whether discrimination occurred.

## Methodology

**Statistical method:** Binary logistic regression (statsmodels.api.Logit). The outcome variable is denial (1) vs. origination (0) per FFIEC exam procedures. The coefficient on the protected class indicator is exponentiated to produce the adjusted odds ratio.

**Regulatory basis:**
- FFIEC Interagency Fair Lending Examination Procedures (August 2009). Federal Financial Institutions Examination Council. (https://www.ffiec.gov/PDF/fairlend.pdf)
- Martinez, Emmanuel and Lauren Kirchner. 'How We Investigated Racial Disparities in Federal Mortgage Data.' The Markup, August 25, 2021.

**Known upward bias from omitted controls:** Wooldridge, J.M. (2019). Introductory Econometrics: A Modern Approach (7th ed.). Cengage Learning. § 3.3: 'Omitted Variable Bias.' Omitting variables correlated with both the outcome and the regressor of interest biases the estimated coefficient away from zero. AUS recommendations and credit scores — the primary omitted variables — are correlated with both race and denial, producing upward bias in the racial disparity coefficient. Results should be interpreted as an upper bound on the unexplained disparity.

**Full methodology documentation:** `docs/methodology.md` in the package repository.

## Reproducibility

The following provenance block contains all information needed to reproduce this analysis. Copy it verbatim if citing this result.

```json
{
  "package_version": "0.2.0",
  "dependency_versions": {
    "statsmodels": "0.14.6",
    "pandas": "2.3.3",
    "numpy": "2.0.2"
  },
  "data_source_url": "https://ffiec.cfpb.gov/v2/data-browser-api/view",
  "timestamp": "2026-05-24T02:32:47.999794+00:00",
  "input_parameters": {
    "protected_class": "Black or African American",
    "comparison_class": "White",
    "controls": [
      "log_income",
      "log_loan_amount",
      "loan_to_value_ratio",
      "log_property_value"
    ],
    "msa": null,
    "min_sample_size": 500,
    "data_year": 2023
  },
  "methodology_reference": "docs/methodology.md",
  "calibration_reference": "The Markup (2021): adjusted OR 1.8\u00d7 for Black vs. White, 2019 HMDA conventional first-lien home purchase. Expected range for this model (without AUS/credit score controls): 1.6\u20132.2\u00d7."
}
```

*Methodology concerns? Open a GitHub issue tagged `methodology` at https://github.com/Jaypatel1511/fair-lending-screener/issues with specific concerns and citations. All methodology changes are versioned in CHANGELOG.md.*


---

**A note on this demo's results vs. the calibration target:**

This state-level analysis produced 0 MSA fixed effects because the filtered sample concentrates in one metropolitan area (Chicago-Naperville-Elgin). With no geographic controls, local market variation — home prices, lender mix, economic conditions — is attributed to race, inflating the adjusted odds ratio.

The adjusted OR here (4.81×) exceeds the 1.6×–2.2× calibration range established by The Markup (2021) on national data with ~300+ MSA dummies. Running on national data would produce MSA controls and bring the OR closer to the calibration target. This is expected behavior, not a bug — it demonstrates why geographic controls matter in fair lending analysis.

---

## 6. What a Non-Significant Result Looks Like

The tool's behavior is conditional on significance — it produces different output depending on whether the adjusted disparity clears the p < 0.05 threshold with a confidence interval that excludes 1.0.

**When the disparity is NOT statistically significant:**

- `result.is_statistically_significant` is `False`
- The headline reads: *"the adjusted denial disparity ... was **not statistically significant**"*
- Lender name is suppressed from the headline (even if passed)
- `result.interpretation` contains the non-significant disclaimer instead of the significant one
- `result.limitations[-1]` is replaced with the non-significant disclaimer

**Why non-significant ≠ exoneration:**

Absence of statistical significance at p < 0.05 in a model that lacks credit score and AUS controls does not prove that no disparity exists. It means the available public data are insufficient to detect one at conventional thresholds. Federal examiners with access to full loan files, credit data, and AUS records — and with the ability to run matched-pair comparative file reviews — may reach a different conclusion on the same lender in the same year.

The non-significant disclaimer is intentionally not an exoneration. Its exact language, which appears automatically in every non-significant result, is shown in the cell below.

In [8]:
# This cell is locally testable — imports a constant, no API call required.

from fair_lending_screener.methodology import (
    STANDARD_DISCLAIMER_NON_SIGNIFICANT,
    STANDARD_DISCLAIMER,
)

print("When is_statistically_significant is True, result.interpretation ends with:")
print()
print(f'  "{STANDARD_DISCLAIMER}"')
print()
print("-" * 70)
print()
print("When is_statistically_significant is False, result.interpretation ends with:")
print()
print(f'  "{STANDARD_DISCLAIMER_NON_SIGNIFICANT}"')

When is_statistically_significant is True, result.interpretation ends with:

  "This analysis identifies a statistically significant adjusted disparity in denial rates. It does not constitute a finding of discrimination under ECOA or the Fair Housing Act, and it does not establish that any protected class characteristic caused any lending outcome. Further review of application files, underwriting guidelines, and internal lender data would be required to assess whether discrimination occurred."

----------------------------------------------------------------------

When is_statistically_significant is False, result.interpretation ends with:

  "This analysis did NOT identify a statistically significant adjusted disparity at the conventional p<0.05 threshold. Absence of statistical significance does not prove the absence of disparity — it may reflect sample size, model specification, or genuinely small disparities. This tool is a screening signal and does not constitute a finding of dis

## 7. Accessing the Bundled Methodology

The full methodology documentation (`_methodology_doc.md`) and limitations documentation (`_limitations_doc.md`) are bundled inside the package wheel at install time. They travel with every installation — there is no separate document that can drift out of sync with the version that produced a given result.

`get_methodology_path()` and `get_limitations_path()` return `pathlib.Path` objects pointing to those files in your local environment. This means you can always open the methodology that corresponds exactly to the version of the tool you ran.

In [9]:
# Locally testable — no API call required.

methodology_path = fls.get_methodology_path()
limitations_path = fls.get_limitations_path()

print(f"Methodology doc: {methodology_path}")
print(f"Limitations doc: {limitations_path}")
print()

# Display the first 20 lines of the methodology doc
print("First 20 lines of docs/methodology.md (bundled in the package):")
print("-" * 60)
with open(methodology_path) as f:
    for i, line in enumerate(f):
        if i >= 20:
            break
        print(line, end="")

Methodology doc: /private/tmp/notebook_test/lib/python3.9/site-packages/fair_lending_screener/_methodology_doc.md
Limitations doc: /private/tmp/notebook_test/lib/python3.9/site-packages/fair_lending_screener/_limitations_doc.md

First 20 lines of docs/methodology.md (bundled in the package):
------------------------------------------------------------
# Fair Lending Screener: Statistical Methodology

**Version:** 0.2.0  
**Last updated:** 2026-05-22  
**Status:** Alpha — methodology peer review by external fair lending expert planned before v1.0.0

---

## Overview

This document describes the statistical methodology used in `fair-lending-screener` for adjusted denial disparity analysis. Every analytical decision made in this package traces to a published regulatory source, academic methodology, or documented industry practice. Where a decision involves judgment, that judgment is explained and the alternative is noted.

The goal of this analysis is to answer a narrow, specific question

## 8. Limitations and Next Steps

### What this analysis cannot tell you

**Credit score is absent from public HMDA data.** The single most predictive variable in mortgage underwriting was excluded from the public dataset during Dodd-Frank rulemaking. Its absence means the adjusted odds ratio from this tool is an upper-bound estimate of the unexplained disparity — a fully specified model with credit score controls would likely produce a lower (but possibly still significant) adjusted odds ratio. The direction of bias is upward per standard omitted-variable bias theory (Wooldridge 2019, § 3.3).

**AUS recommendations are not in public HMDA.** Fannie Mae Desktop Underwriter (DU) and Freddie Mac Loan Prospector (LP) decisions drive the majority of conventional loan underwriting. An applicant who received an AUS "Refer" may be denied regardless of demographics; that signal is invisible in public data.

**Asset and reserve data are not reported.** Liquid reserves are a required or preferred element in many underwriting guidelines and are entirely absent from HMDA.

**This is a screening tool, not a finding.** A statistically significant adjusted disparity means the pattern warrants further review with full loan-file data, internal underwriting guidelines, and examiner access. It does not mean the lender violated ECOA or the Fair Housing Act, and it does not establish that any specific applicant was treated unlawfully.

**Federal examiners may reach a different conclusion.** OCC, Federal Reserve, FDIC, NCUA, and CFPB examiners have access to credit scores (submitted under supervisory authority), full loan files, AUS records, lender overlays, and the ability to conduct comparative file reviews (matched-pair analysis). Their analysis of the same lender, with access to that data, may show higher, lower, or statistically non-significant disparities.

### Next steps if you find a significant disparity

1. Check whether the result survives different sample windows (different years, adjacent geographies)
2. Review the pseudo-R² — a value below 0.10 suggests the model may be underspecified for this lender/market
3. Look at the unadjusted vs. adjusted gap — a large gap means income and LTV explain much of the raw disparity; a small gap means they don't
4. File a complaint with CFPB, HUD, or the appropriate prudential regulator if the pattern is persistent across years and geographies
5. Share findings with a fair housing attorney before publication

---

For questions about the methodology or to report issues, visit the [GitHub repository](https://github.com/Jaypatel1511/fair-lending-screener).